# **Libraries**


In [13]:
from langchain_neo4j import LLMGraphTransformer #  This class is used to extract nodes and relationships from text
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate # It can define system and user messages separately.
from langchain_community.document_loaders import PyPDFLoader # Used to load the PDF document to convert it later to document objects
from langchain_text_splitters import RecursiveCharacterTextSplitter # Used for splitting the documents into smaller chunks (Piece of text) for better processing and analysis

import ollama

from pyvis.network import Network
import os

import torch

# **LLMGraphTransformer Knowledge Graph**


In [14]:
doc_path = r"/home/trainee3/Desktop/GraphRAG/abel-and-his-great-adventure.pdf"

if doc_path: # Checking if the document path exist
    # Initializing the loader
    loader = PyPDFLoader(file_path=doc_path) # Loading the PDF document using PyPDFLoader and creating the document objects
    
    pages = loader.load() #  The pages is a List of document objects, where each document object represents a page in the PDF. 
    
    print(f"Loaded {len(pages)} document object from {doc_path}")
else:
    print("No document path provided.")

# Access content
print(len(pages))
#content = data[0].page_content # Reading the first page of the document 
#print(content[:]) # printing the content of the first page

Loaded 11 document object from /home/trainee3/Desktop/GraphRAG/abel-and-his-great-adventure.pdf
11


In [15]:
splitter = RecursiveCharacterTextSplitter( # Defining the splitter
    chunk_size=3000,      # characters per chunk — tune based on how slow/fast your model runs
    chunk_overlap=200     # each chunk shares its last 200 characters
)

chunks = splitter.split_documents(pages) # Creating chunks
"""
Takes your pages (the per-page Document list from PyPDFLoader) and runs the splitter on each one, producing a new, longer list of smaller Document chunks. 
Each output chunk still keeps its original metadata (like which page it came from), just with page_content broken into smaller pieces.
"""


print(f"Split into {len(chunks)} chunks")
print(chunks[1])

Split into 13 chunks
page_content='He was a tall man, somewhat ungainly of figure and homely of face. But his large, deep eyes of 
velvety nut-brown were very beautiful and marvellously bright and clear for a man of his age. 
He wore a little pointed, well-cared-for beard, innocent of gray; but his hair was grizzled, and 
altogether he had the appearance of a man who had passed through many sorrows which had 
marked his body as well as his soul. Looking at him, I doubted Mrs. Campbell’s conclusion that 
he had not “minded” giving up college. This man had given up much and felt it deeply; but he 
had outlived the pain and the blessing of sacrifice had come to him. His voice was very 
melodious and beautiful, and the brown hand he held out to me was peculiarly long and shapely 
and flexible.
We went out to the garden in the scented moist air of a maritime spring evening. Behind the 
garden was a cloudy pine wood; the house closed it in on the left, while in front and on the 
right a row 

In [16]:
model = ChatOllama(model="gpt-oss:20b") # ChatOllama connects LangChain to a model running locally through Ollama.
#graph_transformer = LLMGraphTransformer(llm=model) # This class uses a language model to convert unstructured text into a knowledge graph structure.

In [17]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """
You extract a clean knowledge graph from a narrative story.

Extraction rules:

1. Extract only information explicitly supported by the passage.
2. Use the canonical character name instead of pronouns.
3. Merge aliases that clearly refer to the same entity.
4. Use singular, title-cased node identifiers.
5. Use uppercase relationship names with underscores.
6. Do not create nodes for adjectives, pronouns, numbers, or generic words.
7. Store descriptive details as node properties where possible.
8. Represent important actions as Event nodes when they:
   - involve multiple entities,
   - change the story state,
   - occur at a specific location, or
   - must be ordered in the narrative.
9. Do not infer unsupported motivations or facts.
10. Preserve the order of important events.
11. Include a short evidence phrase for every extracted relationship.
12. Return only schema-compliant structured output.

Examples of canonicalization:
- "Abel", "the boy", and "he" may refer to Abel.
- "the animal" and "the fox" may refer to the same fox only when the
  passage provides enough evidence.
"""

# LLMGraphTransformer only ever invokes this chain with {"input": text} per chunk,
# so a "story_context" variable can never be supplied and must not be part of the prompt.
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    (
        "human",
        """Extract the knowledge graph from the following passage.

Passage:
{input}
"""
    ),
])

In [18]:
allowed_nodes = [
    "Character",
    "Location",
    "Object",
    "Animal",
    "Event",
    "Goal",
]

allowed_relationships = [
    ("Character", "TRAVELS_TO", "Location"),
    ("Character", "LOCATED_IN", "Location"),
    ("Character", "MEETS", "Character"),
    ("Character", "MEETS", "Animal"),
    ("Character", "POSSESSES", "Object"),
    ("Character", "USES", "Object"),
    ("Character", "PARTICIPATES_IN", "Event"),
    ("Event", "OCCURS_IN", "Location"),
    ("Event", "INVOLVES", "Character"),
    ("Event", "PRECEDES", "Event"),
    ("Character", "HAS_GOAL", "Goal"),
    ("Event", "ACHIEVES", "Goal"),
]

In [19]:
#Another form of LLMGraphTransformer

graph_transformer = LLMGraphTransformer(
    llm=model,
    allowed_nodes = allowed_nodes, 
    allowed_relationships = allowed_relationships,
   node_properties=[
        "name",
        "description",
        "traits",
        "role",
        "aliases",
    ],
    relationship_properties=[
        "description",
        "sequence",
        "evidence",
    ],
    strict_mode=True,
    prompt=prompt
    )

"""The allowed_nodes and allowed_relationships parameters help prevent the model from inventing too many 
different graph labels. """


#graph_documents = graph_transformer.convert_to_graph_documents(chunks)

'The allowed_nodes and allowed_relationships parameters help prevent the model from inventing too many \ndifferent graph labels. '

In [20]:
# documents = [Document(page_content=documents)] # LLMGraphTransformer expects a list of documents

graph_documents = await graph_transformer.aconvert_to_graph_documents(chunks) # It asks the model to independently convert each chunk into nodes and relationships.

 # GraphDocument is a class with three attributes: 1) Nodes: List[Node], 2) relationships: List[Relationship], 3) source: Document

CancelledError: 

In [10]:
all_nodes = []
all_relationships = []
for gd in graph_documents:
    all_nodes.extend(gd.nodes)
    all_relationships.extend(gd.relationships)

print(f"Total: {len(all_nodes)} nodes, {len(all_relationships)} relationships")

Total: 177 nodes, 260 relationships


In [11]:
print(f"Nodes:{graph_documents[0].nodes}")
print(f"Relationships:{graph_documents[0].relationships}")

Nodes:[Node(id='Abel Armstrong', type='Person', properties={}), Node(id='Old Abel', type='Alias', properties={}), Node(id='Stillwater', type='Place', properties={}), Node(id='Board Of School Trustees In Stillwater', type='Organization', properties={}), Node(id='Gray House Close By Harbour Shore', type='Building', properties={}), Node(id='Harbour Shore', type='Place', properties={}), Node(id='Tamzine', type='Person', properties={}), Node(id='Mrs. Campbell', type='Person', properties={}), Node(id='Yaller Cat', type='Animal', properties={}), Node(id='Garden Of Abel', type='Place', properties={}), Node(id='Narrator', type='Person', properties={}), Node(id='L. M. Montgomery', type='Person', properties={}), Node(id='Abel And His Great Adventure', type='Book', properties={})]
Relationships:[Relationship(source=Node(id='Abel Armstrong', type='Person', properties={}), target=Node(id='Board Of School Trustees In Stillwater', type='Organization', properties={}), type='MEMBER_OF', properties={}), 

# **Knowledge Graph Visualization**


In [12]:
from pyvis.network import Network
import os

def visualize_graph(graph_documents):
    net = Network(height="1200px", width="100%", directed=True,
                  notebook=False, bgcolor="#222222", font_color="white")

    nodes = []
    relationships = []
    for gd in graph_documents:
        nodes.extend(gd.nodes)
        relationships.extend(gd.relationships)

    node_dict = {node.id: node for node in nodes}

    # --- Add ALL nodes, not just ones that appear in a relationship ---
    for node in nodes:
        try:
            net.add_node(node.id, label=node.id, title=node.type, group=node.type)
        except:
            continue

    # --- Add edges (unchanged logic — still only valid ones) ---
    for rel in relationships:
        if rel.source.id in node_dict and rel.target.id in node_dict:
            try:
                net.add_edge(rel.source.id, rel.target.id, label=rel.type.lower())
            except:
                continue

    net.set_options("""
    {
        "physics": {
            "forceAtlas2Based": {
                "gravitationalConstant": -100,
                "centralGravity": 0.01,
                "springLength": 200,
                "springConstant": 0.08
            },
            "minVelocity": 0.75,
            "solver": "forceAtlas2Based"
        }
    }
    """)

    output_file = "knowledge_graph.html"
    net.save_graph(output_file)
    print(f"Graph saved to {os.path.abspath(output_file)}")

    try:
        import webbrowser
        webbrowser.open(f"file://{os.path.abspath(output_file)}")
    except:
        print("Could not open browser automatically")

visualize_graph(graph_documents)

Graph saved to /home/trainee3/Desktop/GraphRAG/knowledge_graph.html


# **Extracting Json Files**


In [13]:
import json

# --- Aggregate nodes and relationships across ALL chunks ---
all_nodes = []
all_relationships = []
for gd in graph_documents:
    all_nodes.extend(gd.nodes)
    all_relationships.extend(gd.relationships)

# --- Export nodes to JSON ---
nodes_data = [
    {
        "id": n.id,
        "type": n.type,
        "properties": n.properties or {}
    }
    for n in all_nodes
]

with open("Storynodes.json", "w") as f:
    json.dump(nodes_data, f, indent=2)

# --- Export relationships to JSON ---
relationships_data = [
    {
        "source": r.source.id,   # flatten Node object down to just its id string
        "target": r.target.id,
        "type": r.type,
        "properties": r.properties or {}
    }
    for r in all_relationships
]

with open("Storyrelationships.json", "w") as f:
    json.dump(relationships_data, f, indent=2)

print(f"Exported {len(nodes_data)} nodes to Storynodes2.json")
print(f"Exported {len(relationships_data)} relationships to Storyrelationships2.json")

Exported 177 nodes to Storynodes2.json
Exported 260 relationships to Storyrelationships2.json


In [14]:
import json
from collections import Counter

with open("Storynodes.json") as f:
    nodes = json.load(f)

pairs = [(n["id"], n["type"]) for n in nodes]
counts = Counter(pairs)
duplicates = {k: v for k, v in counts.items() if v > 1}

print(f"Total entries in JSON: {len(nodes)}")
print(f"Unique (id, type) pairs: {len(counts)}")
print(f"Number of duplicated pairs: {len(duplicates)}")
for k, v in list(duplicates.items())[:10]:
    print(k, "→ appears", v, "times")

Total entries in JSON: 177
Unique (id, type) pairs: 133
Number of duplicated pairs: 19
('Stillwater', 'Place') → appears 2 times
('Tamzine', 'Person') → appears 5 times
('Narrator', 'Person') → appears 5 times
('Mrs_Campbell', 'Person') → appears 2 times
('Abel', 'Person') → appears 9 times
('Garden', 'Place') → appears 8 times
('House', 'Place') → appears 2 times
('Moonrise', 'Event') → appears 2 times
('Master', 'Person') → appears 2 times
('Mr. Armstrong', 'Person') → appears 2 times


In [15]:
import json
from collections import defaultdict

with open("Storynodes.json") as f:
    nodes = json.load(f)

id_to_types = defaultdict(set)
for n in nodes:
    id_to_types[n["id"]].add(n["type"])

inconsistent = {id_: types for id_, types in id_to_types.items() if len(types) > 1}
print(f"Entity IDs with more than one type: {len(inconsistent)}")
for id_, types in list(inconsistent.items())[:10]:
    print(id_, "→", types)

Entity IDs with more than one type: 7
Old Abel → {'Person', 'Alias'}
Garden → {'Location', 'Place'}
Hop-Vine Arbour → {'Arbour', 'Place'}
Master → {'Title', 'Person'}
Captain Kidd → {'Cat', 'Person'}
Waves → {'Object', 'Thing'}
Wind → {'Object', 'Thing'}
